In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd
import numpy as np

train_df = pd.read_csv('/kaggle/input/competitions/sentiment-analysis-on-movie-reviews/train.tsv.zip',sep = '\t')
test_df = pd.read_csv('/kaggle/input/competitions/sentiment-analysis-on-movie-reviews/test.tsv.zip',sep = '\t')



In [ ]:
train_df.head()

In [ ]:
train_df.shape

In [ ]:
test_df.shape

In [ ]:
train_df.info()

In [ ]:
train_df['Sentiment'].value_counts()

In [ ]:
test_df.head()

In [ ]:
test_df.shape

In [ ]:
test_df.info()

In [ ]:
X_temp_train = train_df['Phrase'].fillna("").values
y_temp = train_df['Sentiment'].values

X_temp_test= test_df['Phrase'].fillna("").values

In [ ]:
y_temp

In [ ]:
X_temp_train

In [ ]:
from tensorflow.keras.utils import to_categorical
y_train = to_categorical(y_temp, num_classes=5)

In [ ]:
from tensorflow.keras.layers import TextVectorization

vocab_size = 15000
max_length = 50     

vectorizer = TextVectorization(
    max_tokens=vocab_size,
    output_mode='int',
    output_sequence_length=max_length
)

vectorizer.adapt(X_temp_train)

In [ ]:
X_train = vectorizer(X_temp_train)
X_test = vectorizer(X_temp_test)

# Single RNN training

In [ ]:
"""
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense

model1 = Sequential()
model1.add(Input(shape=(max_length,))) 
model1.add(Embedding(input_dim=vocab_size, output_dim=32)) 
model1.add(SimpleRNN(units=64,return_sequences=False))
model1.add(Dense(units=5, activation='softmax')) 

model1.compile(
    optimizer='adam', 
    loss='categorical_crossentropy', 
    metrics=['accuracy']
)
"""

In [ ]:
"""
history = model1.fit(
    X_train, 
    y_train, 
    epochs=10, 
    batch_size=128, 
    validation_split=0.2
)
"""

In [ ]:
"""
import numpy as np
predictions_proba = model1.predict(X_test)
final_predictions = np.argmax(predictions_proba, axis=1)
submission_df = pd.DataFrame({
    'PhraseId': test_df['PhraseId'],
    'Sentiment': final_predictions
})
submission_df.to_csv('submission.csv', index=False)
#51.789
"""

# Multiple RNN training

In [ ]:
"""
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense

model2 = Sequential()

model2.add(Input(shape=(max_length,)))
model2.add(Embedding(input_dim=vocab_size, output_dim=32)) 
model2.add(SimpleRNN(units=32, return_sequences=True))
model2.add(SimpleRNN(units=64, return_sequences=True)) 
model2.add(SimpleRNN(units=128, return_sequences=False))
model2.add(Dense(units=5, activation='softmax'))

model2.compile(
    optimizer='adam', 
    loss='categorical_crossentropy', 
    metrics=['accuracy']
)
"""


In [ ]:
"""
history2 = model2.fit(
    X_train, 
    y_train, 
    epochs=10, 
    batch_size=128, 
    validation_split=0.2
)
"""

In [ ]:
"""
import numpy as np
predictions_proba = model2.predict(X_test)
final_predictions = np.argmax(predictions_proba, axis=1)
submission_df = pd.DataFrame({
    'PhraseId': test_df['PhraseId'],
    'Sentiment': final_predictions
})
submission_df.to_csv('submission.csv', index=False)
#59.808
"""

# Bidirectional RNNs training

In [ ]:
"""
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense
from tensorflow.keras.layers import Bidirectional
from tensorflow.keras.layers import Dropout

model3 = Sequential()
model3.add(Input(shape=(max_length,)))
model3.add(Embedding(input_dim=vocab_size, output_dim=32))
model3.add(Bidirectional(SimpleRNN(units=32, return_sequences=True)))
model3.add(Dropout(0.1))
model3.add(Bidirectional(SimpleRNN(units=64, return_sequences=True)))
model3.add(Dropout(0.2))
model3.add(Bidirectional(SimpleRNN(units=16, return_sequences=False)))
model3.add(Dropout(0.2))
model3.add(Dense(units=5, activation='softmax'))

model3.compile(
    optimizer='adam', 
    loss='categorical_crossentropy', 
    metrics=['accuracy']
)
"""


In [ ]:
"""
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=3, 
    restore_best_weights=True
)

history3 = model3.fit(
    X_train, 
    y_train, 
    epochs=10, 
    batch_size=128, 
    validation_split=0.2,
    callbacks=[early_stop]
)
"""


In [ ]:
"""
import numpy as np
predictions_proba = model3.predict(X_test)
final_predictions = np.argmax(predictions_proba, axis=1)
submission_df = pd.DataFrame({
    'PhraseId': test_df['PhraseId'],
    'Sentiment': final_predictions
})
submission_df.to_csv('submission.csv', index=False)
#60.892

"""

#  LSTMs training

In [ ]:
"""
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense
from tensorflow.keras.layers import Bidirectional
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import LSTM

model4 = Sequential()
model4.add(Input(shape=(max_length,)))
model4.add(Embedding(input_dim=vocab_size, output_dim=32))
model4.add(LSTM(units=32, return_sequences=True))
model4.add(Dropout(0.2))
model4.add(LSTM(units=16, return_sequences=True))
model4.add(Dropout(0.2))
model4.add(LSTM(units=16, return_sequences=False))
model4.add(Dropout(0.2))
model4.add(Dense(units=5, activation='softmax'))

model4.compile(
    optimizer='adam', 
    loss='categorical_crossentropy', 
    metrics=['accuracy'],
)
"""



In [ ]:
"""
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=3, 
    restore_best_weights=True
)


history4 = model4.fit(
    X_train, 
    y_train, 
    epochs=10, 
    batch_size=128, 
    validation_split=0.2,
    callbacks = [early_stop]
)
"""



In [ ]:

"""
import numpy as np
predictions_proba = model4.predict(X_test)
final_predictions = np.argmax(predictions_proba, axis=1)
submission_df = pd.DataFrame({
    'PhraseId': test_df['PhraseId'],
    'Sentiment': final_predictions
})
submission_df.to_csv('submission.csv', index=False)
#62.541
"""

# Bidirectional LSTMs training

In [ ]:
"""
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense
from tensorflow.keras.layers import Bidirectional
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import LSTM

model5 = Sequential()
model5.add(Input(shape=(max_length,)))
model5.add(Embedding(input_dim=vocab_size, output_dim=32))
model5.add(Bidirectional(LSTM(units=32, return_sequences=True)))
model5.add(Dropout(0.3))
model5.add(Bidirectional(LSTM(units=16, return_sequences=True)))
model5.add(Dropout(0.2))
model5.add(Bidirectional(LSTM(units=16, return_sequences=False)))
model5.add(Dropout(0.2))
model5.add(Dense(units=5, activation='softmax'))

model5.compile(
    optimizer='adam', 
    loss='categorical_crossentropy', 
    metrics=['accuracy']
)
"""


In [ ]:
"""
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=3, 
    restore_best_weights=True
)




history5 = model5.fit(
    X_train, 
    y_train, 
    epochs=10, 
    batch_size=128, 
    validation_split=0.2,
    callbacks = [early_stop]
)
"""



In [ ]:
"""
import numpy as np
predictions_proba = model5.predict(X_test)
final_predictions = np.argmax(predictions_proba, axis=1)
submission_df = pd.DataFrame({
    'PhraseId': test_df['PhraseId'],
    'Sentiment': final_predictions
})
submission_df.to_csv('submission.csv', index=False)
#62.764
"""

# GRU training

In [ ]:
"""
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense
from tensorflow.keras.layers import Bidirectional
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import GRU

model6 = Sequential()
model6.add(Input(shape=(max_length,)))
model6.add(Embedding(input_dim=vocab_size, output_dim=32))
model6.add(GRU(units=32, return_sequences=True))
model6.add(Dropout(0.3))
model6.add(GRU(units=16, return_sequences=True))
model6.add(Dropout(0.2))
model6.add(GRU(units=16, return_sequences=False))
model6.add(Dropout(0.2))
model6.add(Dense(units=5, activation='softmax'))

model6.compile(
    optimizer='adam', 
    loss='categorical_crossentropy', 
    metrics=['accuracy']
)
"""


In [ ]:
"""
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=3, 
    restore_best_weights=True
)



history6 = model6.fit(
    X_train, 
    y_train, 
    epochs=10, 
    batch_size=128, 
    validation_split=0.2,
    callbacks = [early_stop]
)
"""


In [ ]:
"""
import numpy as np
predictions_proba = model6.predict(X_test)
final_predictions = np.argmax(predictions_proba, axis=1)
submission_df = pd.DataFrame({
    'PhraseId': test_df['PhraseId'],
    'Sentiment': final_predictions
})
submission_df.to_csv('submission.csv', index=False)
#62.855
"""

# Hyperparameter tuning using keras tuner

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense
from tensorflow.keras.layers import Bidirectional
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import GRU
from tensorflow.keras.optimizers import Adam
import keras_tuner as kt

def build_model(hp):
    model = Sequential()
    model.add(Input(shape=(max_length,)))
    model.add(Embedding(input_dim=vocab_size, output_dim=32))

    hp_units_1 = hp.Int('units_1', min_value=32, max_value=64, step=32)
    model.add(LSTM(units=hp_units_1, return_sequences=True))
    
    hp_dropout_1 = hp.Float('dropout_1', min_value=0.2, max_value=0.4, step=0.1)
    model.add(Dropout(hp_dropout_1))
    hp_units_2 = hp.Int('units_2', min_value=16, max_value=32, step=16)
    model.add(LSTM(units=hp_units_2, return_sequences=True))

    hp_dropout_2 = hp.Float('dropout_2', min_value=0.1, max_value=0.3, step=0.1)
    model.add(Dropout(hp_dropout_2))

    hp_units_3 = hp.Int('units_2', min_value=8, max_value=16, step=8)
    model.add(LSTM(units=hp_units_2, return_sequences=False))

    hp_dropout_3 = hp.Float('dropout_2', min_value=0.1, max_value=0.3, step=0.1)
    model.add(Dropout(hp_dropout_2))

    model.add(Dense(units=5, activation='softmax'))

    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    
    model.compile(
        optimizer=Adam(learning_rate=hp_learning_rate), 
        loss='categorical_crossentropy', 
        metrics=['accuracy']
    )
    
    return model

In [ ]:

tuner = kt.Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=10,          
    factor=3,              
    directory='direc',
    project_name='movie'
)


In [ ]:

from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=2, 
    restore_best_weights=True
)

tuner.search(
    X_train, 
    y_train, 
    epochs=10, 
    batch_size=128, 
    validation_split=0.2,
    callbacks = [early_stop]
)


In [ ]:

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
model7 = tuner.hypermodel.build(best_hps)

history7 = model7.fit(
    X_train, 
    y_train, 
    epochs=15, 
    batch_size=128, 
    validation_split=0.2,
    callbacks=[early_stop]
)


In [ ]:

predictions_proba7 = model7.predict(X_test)
final_predictions7 = np.argmax(predictions_proba7, axis=1)
submission_df = pd.DataFrame({
    'PhraseId': test_df['PhraseId'],
    'Sentiment': final_predictions7
})
submission_df.to_csv('submission.csv', index=False)
#62.966
